# Interactive Dashboard — ipywidgets
**ENGR 010 Group Project — Power Systems Analysis and Monitoring**

This notebook builds an interactive dashboard using `ipywidgets`. Dropdowns, sliders, and buttons update the plots live inside the notebook without rerunning cells. All analysis logic is imported from `analysis.py`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import ipywidgets as widgets
from IPython.display import display, clear_output

import analysis

matplotlib.rcParams['figure.dpi'] = 110
%matplotlib inline

df = analysis.load_data('power_system_data.csv')
df['month'] = df['timestamp'].dt.month
df['hour']  = df['timestamp'].dt.hour

STATIONS  = sorted(df['station_id'].unique().tolist())
PARAM_LABELS = {
    'real_power_mw':       'Real Power (MW)',
    'reactive_power_mvar': 'Reactive Power (MVAR)',
    'voltage_pu':          'Voltage (pu)',
    'current_pu':          'Current (pu)',
    'power_factor':        'Power Factor',
}
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun'}
COLORS = {'SUB_001':'steelblue','SUB_002':'coral','SUB_003':'mediumseagreen'}

print('Data loaded. Dashboard ready.')

Loaded 13,035 records from 'power_system_data.csv'.
Data loaded. Dashboard ready.


---
## Widget 1 — Time Series Explorer

Select a parameter and which stations to display. The plot updates instantly.

In [ ]:
PARAM_OPTIONS = list(PARAM_LABELS.values())                  # display names
PARAM_COLS    = {v: k for k, v in PARAM_LABELS.items()}      # label → column name

param_dd = widgets.Dropdown(
    options=PARAM_OPTIONS,
    value=PARAM_OPTIONS[0],
    description='Parameter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px'),
)

station_checks = widgets.SelectMultiple(
    options=STATIONS,
    value=STATIONS,
    description='Stations:',
    rows=3,
    layout=widgets.Layout(width='180px'),
)

ts_out = widgets.Output()

def update_ts(change=None):
    label    = param_dd.value
    param    = PARAM_COLS[label]
    selected = list(station_checks.value)

    with ts_out:
        clear_output(wait=True)
        if not selected:
            print('Select at least one station.')
            return

        fig, ax = plt.subplots(figsize=(13, 4))
        for sid in selected:
            s = df[df['station_id'] == sid].sort_values('timestamp')
            ax.plot(s['timestamp'], s[param],
                    label=sid, color=COLORS[sid], linewidth=0.7, alpha=0.9)

        ax.set_title(f'{label} — Time Series', fontsize=13)
        ax.set_xlabel('Time')
        ax.set_ylabel(label)
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=25)
        plt.tight_layout()
        plt.show()

param_dd.observe(update_ts, names='value')
station_checks.observe(update_ts, names='value')

controls = widgets.HBox([param_dd, station_checks])
display(controls, ts_out)
update_ts()

Output()

---
## Widget 2 — Station Summary Statistics

Choose a station to see its mean, median, std, min, and max for every measurement.

In [3]:
stat_dd = widgets.Dropdown(
    options=STATIONS,
    description='Station:',
    layout=widgets.Layout(width='200px'),
)

stat_out = widgets.Output()

def update_stats(change=None):
    sid   = stat_dd.value
    stats = analysis.calculate_statistics(df, station_id=sid)

    with stat_out:
        clear_output(wait=True)
        print(f'Statistics for {sid}\n')
        header = f"{'Measurement':<25} {'Mean':>10} {'Median':>10} {'Std':>10} {'Min':>10} {'Max':>10}"
        print(header)
        print('-' * len(header))
        for col, s in stats.items():
            print(f"{col:<25} {s['mean']:>10.4f} {s['median']:>10.4f} "
                  f"{s['std']:>10.4f} {s['min']:>10.4f} {s['max']:>10.4f}")

        # Mini bar chart of mean values for key measurements
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        keys  = ['real_power_mw', 'voltage_pu', 'power_factor']
        names = ['Real Power (MW)', 'Voltage (pu)', 'Power Factor']
        for ax, key, name in zip(axes, keys, names):
            s = stats[key]
            ax.bar(['Mean','Median'], [s['mean'], s['median']],
                   color=COLORS[sid], alpha=0.8, edgecolor='white')
            ax.errorbar(['Mean'], [s['mean']], yerr=[[0],[s['std']]],
                        fmt='none', color='black', capsize=5, lw=1.5)
            ax.set_title(name, fontsize=10)
            ax.grid(True, alpha=0.3, axis='y')
        plt.suptitle(f'{sid} — Key Measurement Summary', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()

stat_dd.observe(update_stats, names='value')
display(stat_dd, stat_out)
update_stats()

Dropdown(description='Station:', layout=Layout(width='200px'), options=('SUB_001', 'SUB_002', 'SUB_003'), valu…

Output()

---
## Widget 3 — Monthly Load Pattern Explorer

Use the slider to pick a month and see that month's average hourly load profile for all three stations.

In [4]:
month_slider = widgets.IntSlider(
    value=1, min=1, max=6, step=1,
    description='Month:',
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width='350px'),
)
month_label = widgets.Label(value=MONTH_NAMES[1])

def update_month_label(change):
    month_label.value = MONTH_NAMES[change['new']]

month_slider.observe(update_month_label, names='value')

pattern_out = widgets.Output()

def update_pattern(change=None):
    m = month_slider.value
    month_df = df[df['month'] == m]

    with pattern_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, 4))
        for sid in STATIONS:
            s = month_df[month_df['station_id'] == sid]
            hourly = s.groupby('hour')['real_power_mw'].mean()
            ax.plot(hourly.index, hourly.values,
                    label=sid, color=COLORS[sid], linewidth=2, marker='o', markersize=3)

        ax.set_title(f'Average Hourly Load — {MONTH_NAMES[m]} 2024', fontsize=12)
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel('Average Real Power (MW)')
        ax.set_xticks(range(0, 24, 2))
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

month_slider.observe(update_pattern, names='value')

display(widgets.HBox([month_slider, month_label]), pattern_out)
update_pattern()

Output()

---
## Widget 4 — Grid Standards Violation Filter

Filter detected violations by station and severity using toggle buttons.

In [5]:
violations = analysis.check_grid_standards(df)

viol_station = widgets.ToggleButtons(
    options=['All'] + STATIONS,
    description='Station:',
    button_style='info',
)
viol_severity = widgets.ToggleButtons(
    options=['All', 'CRITICAL', 'WARNING'],
    description='Severity:',
    button_style='warning',
)

viol_out = widgets.Output()

def update_violations(change=None):
    filtered = violations.copy()
    if viol_station.value != 'All':
        filtered = filtered[filtered['station_id'] == viol_station.value]
    if viol_severity.value != 'All':
        filtered = filtered[filtered['severity'] == viol_severity.value]

    with viol_out:
        clear_output(wait=True)
        if filtered.empty:
            print('No violations match the selected filters.')
        else:
            print(f'{len(filtered):,} violation(s) found:\n')
            summary = (filtered
                       .groupby(['station_id','type','severity'])
                       .size()
                       .reset_index(name='count'))
            display(summary)

viol_station.observe(update_violations, names='value')
viol_severity.observe(update_violations, names='value')

display(viol_station, viol_severity, viol_out)
update_violations()

ToggleButtons(button_style='info', description='Station:', options=('All', 'SUB_001', 'SUB_002', 'SUB_003'), v…

ToggleButtons(button_style='warning', description='Severity:', options=('All', 'CRITICAL', 'WARNING'), value='…

Output()

---
## Widget 5 — Grid Health Score

Select a station to see its composite health score and the breakdown of each component.

In [6]:
health_scores = analysis.calculate_grid_health_score(df)
pqi           = analysis.calculate_power_quality_indices(df)

health_dd = widgets.Dropdown(
    options=STATIONS,
    description='Station:',
    layout=widgets.Layout(width='200px'),
)

health_out = widgets.Output()

def update_health(change=None):
    sid   = health_dd.value
    score = health_scores[sid]
    idx   = pqi[sid]

    if score >= 90:
        status, color = 'GOOD', 'seagreen'
    elif score >= 75:
        status, color = 'FAIR', 'darkorange'
    else:
        status, color = 'POOR', 'crimson'

    with health_out:
        clear_output(wait=True)
        print(f'{sid}  →  Health Score: {score:.1f} / 100   [{status}]\n')

        components = [
            ('Voltage Compliance',    idx['voltage_compliance_pct'],  0.40),
            ('PF Compliance',         idx['pf_compliance_pct'],       0.40),
            ('Load Factor (scaled)',  idx['load_factor'] * 100,       0.20),
        ]

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        # Breakdown bar
        labels  = [c[0] for c in components]
        contrib = [c[1] * c[2] for c in components]
        bar_colors = [color] * len(components)
        axes[0].barh(labels, contrib, color=bar_colors, alpha=0.85, edgecolor='white')
        axes[0].set_xlim(0, 50)
        axes[0].set_xlabel('Weighted Contribution to Score')
        axes[0].set_title(f'{sid} — Score Breakdown')
        axes[0].grid(True, alpha=0.3, axis='x')

        # All-station comparison
        all_stations = list(health_scores.keys())
        all_scores   = list(health_scores.values())
        bar_c = ['seagreen' if s >= 90 else 'darkorange' if s >= 75 else 'crimson'
                 for s in all_scores]
        bars = axes[1].bar(all_stations, all_scores, color=bar_c, alpha=0.85, edgecolor='white')
        for bar, s in zip(bars, all_scores):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                         f'{s:.1f}', ha='center', fontweight='bold')
        axes[1].set_ylim(0, 108)
        axes[1].set_ylabel('Health Score')
        axes[1].set_title('All Substations Comparison')
        axes[1].axhline(90, color='seagreen',   linestyle='--', lw=1.2, label='Good')
        axes[1].axhline(75, color='darkorange', linestyle='--', lw=1.2, label='Fair')
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

health_dd.observe(update_health, names='value')
display(health_dd, health_out)
update_health()

Dropdown(description='Station:', layout=Layout(width='200px'), options=('SUB_001', 'SUB_002', 'SUB_003'), valu…

Output()